# Retrieve Training Data from GraphDB

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON


def _get_query_results(repo):
    try:
        # specify the repository
        sparql = SPARQLWrapper(f'{repo}')

        # query => retrieving data
        query = '''
            PREFIX data: <http://purl.org/spatialai/onner/onner-full/data#>
            PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

            SELECT ?paragraph ?entity ?offset ?length ?label ?status
            WHERE {
                ?entId rdf:type onner:LabeledTerm ;
                       onner:labeledTermText ?entity ;
                       onner:offset ?offset ;
                       onner:length ?length ;
                       onner:labeledTermDirectlyContainedBy ?paraId ;
                       onner:hasLabeledTermStatus ?status .

                ?paraId onner:paragraphText ?paragraph .

                ?status onner:statusAssignmentDate ?datetime ;
                        onner:hasLabeledTermLabel ?labelId .

                ?labelId onner:labelText ?label .

                FILTER NOT EXISTS {
                    ?entId onner:hasLabeledTermStatus ?newerStatus .
                    ?newerStatus onner:statusAssignmentDate ?newerDatetime .

                    FILTER(?newerDatetime > ?datetime)
                }

                FILTER(
                    !STRSTARTS(STR(?status), STR(data:Rejected_)) &&
                    !STRSTARTS(STR(?status), STR(data:Candidate_))
                )
            }
            ORDER BY ?paraId ?offset
        '''

        sparql.setQuery(query)              # set query
        sparql.setReturnFormat(JSON)        # convert results to json
        results = sparql.query().convert()  # execute query

        return results

    except Exception as e:
        print(f'Error querying the SPARQL endpoint: {e}')
        return None

In [2]:
def get_training_data(repo):
    results = _get_query_results(repo)
    records = results['results']['bindings']
    length = len(records)
    start_index = 0
    index_ranges = []
    
    for i in range(length-1):
        if records[i]['paragraph']['value'] != records[i+1]['paragraph']['value']:
            end_index = i + 1
            index_ranges.append([start_index, end_index])
            start_index = end_index
    
    index_ranges.append([start_index, len(records)])
    
    annotations = []
    
    for range_ in index_ranges:
        start_index = range_[0]
        end_index = range_[1]
        paragraph = records[start_index]['paragraph']['value']
        entities = []
        
        for j in records[start_index:end_index]:
            entity = j['entity']['value']
            offset = j['offset']['value']
            length = j['length']['value']
            label = j['label']['value']
            status = j['status']['value']
    
            start_span = int(offset)
            end_span = start_span + int(length)
            entities.append([start_span, end_span, label])
    
        annotations.append([paragraph, {'entities': entities}])
    
    return annotations


In [3]:
import json
from datetime import datetime


def main(): 
    graphdb_repo = 'http://dev:7200/repositories/GetTrainData' 
    spacy_data = get_training_data(graphdb_repo)
    timestamp = datetime.now().strftime('%y%m%d%H%M%S')
    
    with open(f'training_data_spacy_{timestamp}.json', 'w', encoding='utf-8') as f:
        json.dump(spacy_data, f, indent=4, ensure_ascii=False)
        

In [4]:
if __name__ == '__main__':
    main()

In [20]:
# TEST CELL

graphdb_repo = 'http://dev:7200/repositories/Demon_FOIS' 
generate_train_data_spacy(graphdb_repo)

[['Inter-nanoparticle forces and interaction energy play an important role in the preparation of polymer nanocomposite systems. The magnitude of van der Waals interaction energy and forces between two isolated nanoparticles depends on the geometry of nanoparticles, as well as inter-particle distance. CNC can be considered as a cylindrical nanoparticle with 7 nm radius and 150 nm length . Table  shows equations and estimates for inter-particle interaction energy and force between two cross and parallel CNC cylindrical particles. U <sub>i</sub> , F <sub>i</sub> , A <sub>H</sub> , R, L, and z are the van der Waal interaction energy between particles, inter-particle force, Hamaker constant, particle radius, cylinder length, and interparticle distance, respectively. A <sub>H</sub> , the Hamaker constant, is defined as : where D <sub>0</sub> and γ are the theoretical molecular distance between two particles in contact, known as "cutoff", 0.165 nm, and surface energy of CNC at 190 °C , respec